In [1]:
import torch
import torch.nn as nn

In [ ]:
class ComplexBatchNorm2D(nn.Module):
    def __init__(self, num_features, momentum=0.1, eps=1e-5, affine=True, track_running_stats=True):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        # keep afine always set to True; makes the shifting and scaling after whitening/normalization
        self.affine = affine
        self.momentum = momentum
        self.track_running_stats = track_running_stats
        
        if affine:
            # Scale and shift parameters for complex numbers
            # gamma is a 2x2 matrix per channel
            self.gamma = nn.Parameter(torch.eye(2).repeat(num_features, 1, 1))  # [C, 2, 2]
            self.beta = nn.Parameter(torch.zeros(num_features, 2))              # [C, 2]
        else:
            self.register_parameter("gamma", None)
            self.register_parameter("beta", None)

        if self.track_running_stats:
            # Running statistics  - mean and covariance matrix
            self.register_buffer("running_mean", torch.zeros(num_features, 2))
            # 2x2 covariance matrix per channel
            self.register_buffer("running_cov", torch.eye(2).repeat(num_features, 1, 1))

            self.register_buffer('num_batches_tracked', torch.tensor(0, dtype=torch.long))
        else:
            self.register_buffer('running_mean', None)
            self.register_buffer('running_cov', None)
            self.register_buffer('num_batches_tracked', None)
        
        self.reset_parameters()
    
    def reset_running_stats(self):
        """Reset running statistics to initial values."""
        if self.track_running_stats:
            self.running_mean.zero_()
            # initialize with unit variance I
            self.running_cov.zero_()
            # Set diagonal to 1 (unit variance for both real and imag)
            self.running_cov[:, 0, 0] = 1.0  # Var(real) = 1
            self.running_cov[:, 1, 1] = 1.0  # Var(imag) = 1
            # Off-diagonal stays 0 (no correlation initially)

            self.num_batches_tracked.zero_()
    
    def reset_parameters(self):
        """Reset learnable parameters and running statistics."""
        self.reset_running_stats()
        if self.affine:
            # Initialize weight matrix as IDENTITY
            self.gamma.data.zero_()
            self.gamma.data[:, 0, 0] = 1.0  # w_rr = 1
            self.gamma.data[:, 1, 1] = 1.0  # w_ii = 1
            # Off-diagonal stays 0 (w_ri = w_ir = 0) no correlation 
        
            # Initialize bias as ZEROS
            self.beta.data.zero_()

    def _check_input_dim(self, x: torch.Tensor):
        """Verify input has correct dimensions."""
        if x.dim() != 4:
            raise ValueError(f'Expected 4D input (got {x.dim()}D input)')
        if not x.is_complex():
            raise ValueError('Expected complex-valued input tensor')

    def forward(self, x):
        """
        Apply complex batch normalization.
        Args:
            input: Complex tensor of shape (N, C, H, W)
        Returns:
            Normalized complex tensor of same shape
        """
        self._check_input_dim(x)

        # Determine if we should use running stats or compute batch stats
        exponential_average_factor = 0.0
        
        if self.training and self.track_running_stats:
            if self.num_batches_tracked is not None:
                self.num_batches_tracked += 1
                if self.momentum is None:  # Use cumulative moving average
                    exponential_average_factor = 1.0 / float(self.num_batches_tracked)
                else:  # Use exponential moving average
                    exponential_average_factor = self.momentum
        
        # stack along the last axis
        x = torch.stack([x.real, x.imag], dim=-1)   # B, C, H, W, 2
        B, C, Hg, Wd, _ = x.shape

        x_perm = x.permute(1, 0, 2, 3, 4)       # [C, B, H, W, 2]
        x_perm = x_perm.reshape(C, -1, 2)       # [C, N, 2]  where N = B*H*W
        
        if self.training or not self.track_running_stats:
            # ---- Compute batch statistics ----
            # Compute mean
            mean = x_perm.mean(dim=1, keepdim=True)  # [C,1,2]
            x_centered = x_perm - mean               # [C,N,2]
            
            # Compute covariance
            cov = torch.zeros(C, 2, 2, dtype=x_perm.dtype, device=x_perm.device)
            for c in range(C):
                xc = x_centered[c]  # [N,2]
                cov[c] = (xc.T @ xc) / xc.shape[0] + self.eps * torch.eye(2, dtype=x_perm.dtype, device=x_perm.device)

            # ---- Update running statistics ----
            if self.training and self.track_running_stats:
                with torch.no_grad():
                  self.running_mean = (
                        exponential_average_factor * mean + 
                        (1 - exponential_average_factor) * self.running_mean
                  )
                  self.running_cov = (
                      exponential_average_factor * cov + 
                      (1 - exponential_average_factor) * self.running_cov
                  )
        else:
            # ---- Use stored running statistics ----
            mean = self.running_mean
            cov = self.running_cov
            x_centered = x_perm - mean     

        # Whitening using Cholesky decomposition
        # Covariance matrix: [[Vrr, Vri], [Vri, Vii]]
        # We need to compute the inverse square root of this matrix
        # x_norm_list = []
        # for c in range(C):
        #     # Use eigen-decomposition
        #     eigvals, eigvecs = torch.linalg.eigh(cov[c])
        #     D_inv_sqrt = torch.diag(eigvals.pow(-0.5))
        #     W = eigvecs @ D_inv_sqrt @ eigvecs.T
        #     x_norm_list.append(x_centered[c] @ W.T)
        # x_norm = torch.stack(x_norm_list, dim=0)
        # print(f"{x_norm = }")
        
        eigvals, eigvecs = torch.linalg.eigh(cov)  # eigvals: [C, 2], eigvecs: [C, 2, 2]
        # Compute D_inv_sqrt: [C, 2, 2] diagonal matrices with (eigvals + eps)^(-0.5)
        D_inv_sqrt = torch.diag_embed((eigvals + self.eps).pow(-0.5))  # [C, 2, 2]
        # W = eigvecs @ D_inv_sqrt @ eigvecs.T for each channel
        # Using batch matrix multiplication: [C, 2, 2] @ [C, 2, 2] @ [C, 2, 2]
        W = torch.bmm(torch.bmm(eigvecs, D_inv_sqrt), eigvecs.transpose(1, 2))  # [C, 2, 2]
        # Apply whitening: x_centered [C, N, 2] @ W.T [C, 2, 2] -> [C, N, 2]
        x_norm = torch.bmm(x_centered, W.transpose(1, 2))  # [C, N, 2]
        
        # print(f"{x_norm1 = }")
        
        # Affine transform
        if self.affine:
            # x_norm = torch.stack([x_norm[c] @ self.gamma[c].T + self.beta[c] for c in range(C)], dim=0)
            x_norm = torch.bmm(x_norm, self.gamma.transpose(1, 2)) + self.beta.unsqueeze(1)  # [C, N, 2]
 
        # Reshape back
        x_norm = x_norm.reshape(C, B, Hg, Wd, 2)
        x_norm = x_norm.permute(1, 0, 2, 3, 4)    # [B,C,H,W,2]

        x_complex = torch.complex(x_norm[..., 0], x_norm[..., 1])
        return x_complex


In [6]:
# Demonstration
print("ComplexBatchNorm2d with Full Matrix Representation\n" + "="*70)

# Create layer
bn = ComplexBatchNorm2D(num_features=4)
print(f"Layer: {bn}\n")

# Show parameter shapes
print("Parameter Shapes:")
print(f"  weight: {bn.gamma.shape}")
print(f"  bias: {bn.beta.shape}")
print(f"  running_mean: {bn.running_mean.shape}")
print(f"  running_cov: {bn.running_cov.shape}")

# Show initial weight matrix
print("\n" + "="*70)
print("Initial Weight Matrix (Identity)")
print("="*70)
for i in range(min(2, bn.num_features)):
    print(f"\nFeature {i}:")
    print(f"  weight[{i}] = [[{bn.gamma[i, 0, 0]:.4f}, {bn.gamma[i, 0, 1]:.4f}],")
    print(f"                  [{bn.gamma[i, 1, 0]:.4f}, {bn.gamma[i, 1, 1]:.4f}]]")
    print(f"  bias[{i}] = [{bn.beta[i, 0]:.4f}, {bn.beta[i, 1]:.4f}]")

print("\nInterpretation:")
print("  - Identity weight matrix means no transformation initially")
print("  - During training, network learns optimal transformation")

# Show initial covariance
print("\n" + "="*70)
print("Initial Covariance Matrix (Identity)")
print("="*70)
for i in range(min(2, bn.num_features)):
    print(f"\nFeature {i}:")
    print(f"  running_cov[{i}] = [[{bn.running_cov[i, 0, 0]:.4f}, {bn.running_cov[i, 0, 1]:.4f}],")
    print(f"                       [{bn.running_cov[i, 1, 0]:.4f}, {bn.running_cov[i, 1, 1]:.4f}]]")

# Test forward pass
print("\n" + "="*70)
print("Forward Pass Test")
print("="*70)

bn.train()
x = torch.randn(8, 4, 16, 16, dtype=torch.complex64)
y = bn(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {y.shape}")

# Count parameters
total_params = sum(p.numel() for p in bn.parameters() if p.requires_grad)
print(f"\nTotal learnable parameters: {total_params}")
print(f"  Weight matrix: {bn.gamma.numel()} (4 per feature)")
print(f"  Bias vector: {bn.beta.numel()} (2 per feature)")
print(f"  Total per feature: 6")

# After training, weight might look different
print("\n" + "="*70)
print("After Training (Simulated)")
print("="*70)

# Simulate some gradient updates
optimizer = torch.optim.SGD(bn.parameters(), lr=0.1)
for _ in range(5):
    x_train = torch.randn(16, 4, 16, 16, dtype=torch.complex64)
    y_train = bn(x_train)
    loss = y_train.abs().mean()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

print("Weight matrix after training (feature 0):")
print(f"  weight[0] = [[{bn.gamma[0, 0, 0]:.4f}, {bn.gamma[0, 0, 1]:.4f}],")
print(f"                [{bn.gamma[0, 1, 0]:.4f}, {bn.gamma[0, 1, 1]:.4f}]]")
print(f"  bias[0] = [{bn.beta[0, 0]:.4f}, {bn.beta[0, 1]:.4f}]")

print("\nNote: Weight has learned to scale/rotate the normalized features")

print("\n✓ Test completed!")


ComplexBatchNorm2d with Full Matrix Representation
Layer: ComplexBatchNorm2D()

Parameter Shapes:
  weight: torch.Size([4, 2, 2])
  bias: torch.Size([4, 2])
  running_mean: torch.Size([4, 2])
  running_cov: torch.Size([4, 2, 2])

Initial Weight Matrix (Identity)

Feature 0:
  weight[0] = [[1.0000, 0.0000],
                  [0.0000, 1.0000]]
  bias[0] = [0.0000, 0.0000]

Feature 1:
  weight[1] = [[1.0000, 0.0000],
                  [0.0000, 1.0000]]
  bias[1] = [0.0000, 0.0000]

Interpretation:
  - Identity weight matrix means no transformation initially
  - During training, network learns optimal transformation

Initial Covariance Matrix (Identity)

Feature 0:
  running_cov[0] = [[1.0000, 0.0000],
                       [0.0000, 1.0000]]

Feature 1:
  running_cov[1] = [[1.0000, 0.0000],
                       [0.0000, 1.0000]]

Forward Pass Test
Input shape: torch.Size([8, 4, 16, 16])
Output shape: torch.Size([8, 4, 16, 16])

Total learnable parameters: 24
  Weight matrix: 16 (4 per f

In [27]:
## another demonstration
import numpy as np
import torch.nn as nn

def from_numpy(numpy):
    r"""Create a complex tensor from numpy array."""
    re = torch.from_numpy(numpy.real)
    im = torch.from_numpy(numpy.imag)
    return torch.complex(re, im)

# test_cplx_batchnorm(random_state):
random_state = np.random.RandomState(42)
randn = random_state.randn

real = torch.randn((16, 64, 64, 64), dtype=torch.float32)
imag = torch.randn((16, 64, 64, 64), dtype=torch.float32) * 1e-1
z = torch.complex(real, imag)


complex_batch_norm = ComplexBatchNorm2D(32)

out = complex_batch_norm(z).detach().numpy()

re, im = out.real, out.imag
assert np.isclose(float(re.mean()), 0.0, atol=1e-1)
assert np.isclose(float(im.mean()), 0.0, atol=1e-1)
assert np.isclose(float((re * re).mean()), 1.0, atol=1e-1)
assert np.isclose(float((im * im).mean()), 1.0, atol=1e-1)
assert np.isclose(float((re * im).mean()), 0.0, atol=1e-1)
print(f"All is asserted")

RuntimeError: The size of tensor a (64) must match the size of tensor b (32) at non-singleton dimension 0

In [ ]:
from torch.nn import Parameter

class ComplexGroupNorm(nn.Module):
    """
    Group Normalization for complex-valued tensors using true complex
    normalization via 2x2 covariance matrix whitening.

    Each group is whitened using the analytic inverse square root of its
    2x2 covariance matrix [[Vrr, Vri], [Vri, Vii]], preserving the
    relationship between real and imaginary components.

    Args:
        num_groups  (int)  : Number of groups to divide channels into.
        num_channels (int) : Number of channels in the input.
        eps         (float): Small value for numerical stability. Default: 1e-5.
        affine      (bool) : If True, learns affine parameters. Default: True.
    """

    def __init__(
        self,
        num_channels: int,
        num_groups: int=32,
        eps: float = 1e-5,
        affine: bool = True,
    ):
        super().__init__()

        if num_channels % num_groups != 0:
            raise ValueError(
                f"num_channels ({num_channels}) must be divisible by "
                f"num_groups ({num_groups})"
            )

        self.num_groups = num_groups
        self.num_channels = num_channels
        self.eps = eps
        self.affine = affine

        if affine:
            self.weight_real = Parameter(torch.ones(num_channels))
            self.weight_imag = Parameter(torch.ones(num_channels))
            self.bias_real   = Parameter(torch.zeros(num_channels))
            self.bias_imag   = Parameter(torch.zeros(num_channels))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Complex tensor of shape (B, C, *spatial) with dtype torch.cfloat
               or torch.cdouble.

        Returns:
            Normalized complex tensor of the same shape and dtype.
        """
        if not x.is_complex():
            raise TypeError(f"Expected a complex tensor, got {x.dtype}")

        real, imag = x.real, x.imag                        # (B, C, *spatial)
        B, C = real.shape[:2]
        spatial = real.shape[2:]
        G  = self.num_groups
        CG = C // G

        # ── reshape into groups: (B, G, C//G, *spatial) ──────────────────────
        real_g = real.view(B, G, CG, *spatial)
        imag_g = imag.view(B, G, CG, *spatial)

        # ── per-group mean (reduce over C//G and all spatial dims) ────────────
        reduce_dims = list(range(2, real_g.ndim))           # [2, 3, ...]
        mu_r = real_g.mean(dim=reduce_dims, keepdim=True)
        mu_i = imag_g.mean(dim=reduce_dims, keepdim=True)
        
        r = real_g - mu_r
        i = imag_g - mu_i

        # ── 2x2 covariance matrix entries ─────────────────────────────────────
        Vrr = (r * r).mean(dim=reduce_dims, keepdim=True) + self.eps
        Vii = (i * i).mean(dim=reduce_dims, keepdim=True) + self.eps
        Vri = (r * i).mean(dim=reduce_dims, keepdim=True)

        # ── analytic inverse square root of the 2x2 symmetric PD matrix ──────
        # Given M = [[Vrr, Vri], [Vri, Vii]], we want M^{-1/2}.
        # Let  τ = sqrt(det(M)) = sqrt(Vrr*Vii - Vri²)
        #      s = sqrt(τ * (Vrr + Vii + 2τ))   [= sqrt(tr(M + τI) * τ)]
        # Then M^{-1/2} = (1/s) * [[Vii+τ, -Vri], [-Vri, Vrr+τ]]
        tau = (Vrr * Vii - Vri ** 2).clamp(min=0).sqrt()
        s   = (tau * (Vrr + Vii + 2 * tau)).clamp(min=self.eps).sqrt()

        real_norm = (r * (Vii + tau) - i * Vri) / s
        imag_norm = (i * (Vrr + tau) - r * Vri) / s

        # ── reshape back to (B, C, *spatial) ─────────────────────────────────
        real_norm = real_norm.view(B, C, *spatial)
        imag_norm = imag_norm.view(B, C, *spatial)

        # ── learnable affine transform ────────────────────────────────────────
        if self.affine:
            shape = (1, C) + (1,) * len(spatial)
            real_norm = real_norm * self.weight_real.view(shape) + self.bias_real.view(shape)
            imag_norm = imag_norm * self.weight_imag.view(shape) + self.bias_imag.view(shape)

        return torch.complex(real_norm, imag_norm)

    def extra_repr(self) -> str:
        return (
            f"num_groups={self.num_groups}, num_channels={self.num_channels}, "
            f"eps={self.eps}, affine={self.affine}"
        )
    



In [34]:
group_norm = ComplexGroupNorm(num_channels=64, num_groups=32)
x_norm = group_norm.forward(z)

real.shape = torch.Size([16, 64, 64, 64]), imag.shape = torch.Size([16, 64, 64, 64]), B = 16, C = 64, G = 32, CG = 2, spatial = torch.Size([64, 64])
torch.Size([16, 32, 2, 64, 64]) torch.Size([16, 32, 2, 64, 64])
mu_r.shape = torch.Size([16, 32, 1, 1, 1]), mu_i.shape = torch.Size([16, 32, 1, 1, 1])
